# Gold layer EDA

## Step 1 verify tables
- verify tables in gold layer.
- check if we need to add something to silver.

## Step 2 create views
- Create atleast two views for distance or length

## Step 3 Marts EDA

### Distance races (mart_distance_event)
- Top 10 fastest times per distance
- Average speed by gender
- Average speed by country
- Performance trends over years
- Most popular distances

### Time races (mart_time_event)
- Most distance covered in 24h
- Top performers by gender
- Country comparison

### General
- Most active countries
- Gender distribution over years
- Age distribution of finishers
- Most popular events

In [0]:
df = spark.table("marathos.silver.obt_marathos")

df.printSchema()

In [0]:
# Step 1
spark.sql("SHOW TABLES IN marathos.gold").display()

In [0]:
# Check row counts
for table in ["fct_results", "dim_event", "dim_athlete", "dim_date"]:
    count = spark.table(f"marathos.gold.{table}").count()
    print(f"{table}: {count}")

In [0]:
# Check schemas
for table in ["fct_results", "dim_event", "dim_athlete", "dim_date"]:
    print(f"\n=== {table} ===")
    spark.table(f"marathos.gold.{table}").printSchema()

In [0]:
%sql
SELECT * FROM marathos.gold.mart_distance_event

## Checking for dupes

In [0]:
from pyspark.sql.functions import col

# Check duplicates in dim_event
dup_events = spark.table("marathos.gold.dim_event").groupBy("event_id").count().where(col("count") > 1)
print(f"Duplicate event_ids: {dup_events.count()}")

# Check duplicates in dim_athlete  
dup_athletes = spark.table("marathos.gold.dim_athlete").groupBy("athlete_id_hash").count().where(col("count") > 1)
print(f"Duplicate athlete_id_hashes: {dup_athletes.count()}")

In [0]:
from pyspark.sql.functions import col

(spark.table("marathos.gold.dim_athlete")
    .groupBy("athlete_id_hash")
    .count()
    .where(col("count") > 1)
    .join(spark.table("marathos.gold.dim_athlete"), "athlete_id_hash")
    .orderBy("count", ascending=False)
    .limit(20)
    .display()
)

## EDA on marts

### Distance races (mart_distance_event)
- Top 10 fastest times per distance
- Average speed by gender
- Average speed by country
- Performance trends over years
- Most popular distances

### Time races (mart_time_event)
- Most distance covered in 24h
- Top performers by gender
- Country comparison

### General
- Most active countries
- Gender distribution over years
- Age distribution of finishers
- Most popular events

### Distance races (mart_distance_event)

In [0]:
%sql
SELECT
    MIN(performance_seconds) AS shortest_time,
    MAX(performance_seconds) AS longest_time,
    MIN(average_speed) AS slowest_speed,
    ROUND(AVG(average_speed), 2) AS average_speed,
    MAX(average_speed) AS fastest_speed,
    event_distance_km
FROM marathos.gold.mart_distance_event
GROUP BY
    event_distance_km
ORDER BY
    event_distance_km ASC;

#### Top 10 fastest times per distance

In [0]:
%sql
SELECT
  athlete_age,
  athlete_gender,
  average_speed,
  event_distance_km,
  event_name
FROM
  marathos.gold.mart_distance_event
WHERE
  athlete_age IS NOT NULL
ORDER BY
  average_speed DESC
LIMIT 10

#### Average speed by gender

In [0]:
%sql
SELECT 
    ROUND(AVG(average_speed), 2) AS avg_speed,
    athlete_gender
FROM marathos.gold.mart_distance_event
WHERE athlete_gender != 'Missing'
GROUP BY athlete_gender

#### Average speed by country

In [0]:
%sql
SELECT 
    MIN(average_speed) AS slowest_speed,
    MAX(average_speed) AS fastest_speed,
    ROUND(AVG(average_speed), 2) AS avg_speed,
    country_name
FROM marathos.gold.mart_distance_event
GROUP BY country_name
ORDER BY avg_speed DESC

#### Most popular distances

In [0]:
%sql
SELECT 
    event_distance_or_length,
    distance_type,
    COUNT(*) AS number_of_races
FROM marathos.gold.mart_distance_event
GROUP BY event_distance_or_length, distance_type
ORDER BY number_of_races DESC
LIMIT 20

### Time races (mart_time_event)
- Most distance covered in 24h
- Top performers by gender
- Country comparison

#### Most distance covered in 24h

In [0]:
%sql
SELECT 
    event_name,
    athlete_gender,
    country_name,
    MAX(performance_distance) AS max_distance_km
FROM marathos.gold.mart_time_event
WHERE event_duration_hours = 24
GROUP BY event_name, athlete_gender, country_name
ORDER BY max_distance_km DESC
LIMIT 20

#### Top performers by gender

In [0]:
%sql
SELECT 
    athlete_gender,
    country_name,
    ROUND(AVG(average_speed), 2) AS avg_speed,
    ROUND(MAX(performance_distance), 2) AS max_distance_km,
    COUNT(*) AS number_of_races
FROM marathos.gold.mart_time_event
WHERE athlete_gender != 'Missing'
GROUP BY athlete_gender, country_name
ORDER BY avg_speed DESC
LIMIT 20

#### Country comparison

In [0]:
%sql
SELECT 
    country_name,
    ROUND(AVG(average_speed), 2) AS avg_speed,
    ROUND(AVG(performance_distance), 2) AS avg_distance_km,
    COUNT(*) AS number_of_races
FROM marathos.gold.mart_time_event
WHERE country_name IS NOT NULL
GROUP BY country_name
ORDER BY avg_speed DESC
LIMIT 20

### General
- Most active countries
- Gender distribution over years
- Age distribution of finishers
- Most popular events

#### Most active countries

In [0]:
%sql
SELECT 
    country_name,
    COUNT(*) AS number_of_finishers
FROM marathos.gold.mart_general
WHERE country_name IS NOT NULL
GROUP BY country_name
ORDER BY number_of_finishers DESC
LIMIT 20

#### Gender distribution over years

In [0]:
%sql
SELECT 
    year_of_event,
    athlete_gender,
    COUNT(*) AS number_of_finishers
FROM marathos.gold.mart_general
WHERE athlete_gender != 'Missing'
GROUP BY year_of_event, athlete_gender
ORDER BY year_of_event, athlete_gender

#### Age distribution of finishers

In [0]:
%sql
SELECT 
    athlete_age,
    COUNT(*) AS number_of_finishers
FROM marathos.gold.mart_general
WHERE athlete_age IS NOT NULL
GROUP BY athlete_age
ORDER BY athlete_age

#### Most popular events

In [0]:
%sql
SELECT 
    event_name,
    distance_type,
    COUNT(*) AS number_of_finishers,
    COUNT(DISTINCT year_of_event) AS years_held
FROM marathos.gold.mart_general
GROUP BY event_name, distance_type
ORDER BY number_of_finishers DESC
LIMIT 20